## This is a sample pandas project that exemplifies how to implement web scraping and crawler, specifically for data science and machine learning purposes, using Python, Pandas, time, requests and BeautifulSoup, along with Anaconda managed environments

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from io import StringIO

In [2]:
# At this project, we how 3 ways of scraping currency conversion rates from realtime sites and displaying
# the best rate at the end (for the tourist who's buying it):
# - getting the rate from an API as a JSON;
# - getting the rate from an ordinary HTML;
# - getting the rate form a tabular HTML.

In [3]:
HEAD = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
result = []
def to_float(text):
    return float(str(text).replace(".", "").replace(",", "."))

In [4]:
try:
    url = "https://wise.com/rates/live?source=USD&target=BRL"
    value = requests.get(url, headers=HEAD, timeout=15).json()["value"]
    value = float(value)
    print(f"Wise - USD->BRL (commercial): R$ {value:.4f}")
    result.append({"source": "Wise", "type": "commercial", "price_sell_usd": value})
except Exception as e:
    print("Failure when connecting to Wise:", e)

# getting currency conversion rate from USD to BRL from WISE url API in json format, just by using
# requests. No need to scrap or crawl any HTML page

Wise - USD->BRL (commercial): R$ 5.1096


In [5]:
try:
    url = "https://dolarhoje.com/dolar-turismo/"
    html = requests.get(url, headers=HEAD, timeout=15).text
    soup = BeautifulSoup(html, "lxml")

    field = soup.find("input", id="nacional") # the HTML element with the rate value
    value = to_float(field["value"])

    print(f"DolarHoje - USD->BRL (tourism): R$ {value:.4f}")
    result.append({"source": "DolarHoje", "type": "tourism", "price_sell_usd": value})
except Exception as e:
    print("Failure when connecting to DolarHoje:", e)

# getting currency conversion rate from USD to BRL from DolarHoje page, in HTML format, using requests
# to get the HTML text and BeautifulSoup to scrap it and get the value from the specific HTML element

DolarHoje - USD->BRL (tourism): R$ 5.3200


In [6]:
try:
    url = "https://www.x-rates.com/table/?from=USD&amount=1"
    html = requests.get(url, headers=HEAD, timeout=15).text

    tables = pd.read_html(StringIO(html))  # returns a list of DataFrames from the HTML tables
    print("Tables found at the HTML page:", len(tables))

    value = None
    for t in tables:
        txt = t.astype(str)
        mask = txt.apply(lambda c: c.str.contains("Brazilian", case=False, na=False)).any(axis=1)
        if mask.any():
            line = t[mask].iloc[0].tolist()
            for x in line:
                try:
                    n = float(x)
                    if 1 < n < 100:
                        value = n
                        break
                except (ValueError, TypeError):
                    pass
            break
    if value:
        print(f"x-rates - USD->BRL (commercial): R$ {value:.4f}")
        result.append({"source": "x-rates", "type": "commercial", "price_sell_usd": value})
    else:
        print("BRL line not found at the table.")
except Exception as e:
    print("Failure when connecting to x-rates:", e)

# getting currency conversion rate from USD to BRL from x-rates page, in HTML table format, using requests
# and then pd.read_html() to scrap it into a Pandas Dataframe. Later filtering the BRL row and finding the 
# rate value

Tables found at the HTML page: 2
x-rates - USD->BRL (commercial): R$ 5.1123


In [7]:
df = pd.DataFrame(result).sort_values("price_sell_usd").reset_index(drop=True)
print("Collected rates (R$ per US$), from the cheapest to the most expensive selling rates:\n")
print(df.to_string(index=False))

df.to_csv("rates_brl_dolar.csv", index=False)

# best rate for the tourist who buys the dolars (USD) with reais (BRL)
tourism = df[df["type"] == "tourism"]
if not tourism.empty:
    best = tourism.iloc[0]
    print(f"\n>>> USD (tourism): {best['source']} at R$ {best['price_sell_usd']:.4f}")

Collected rates (R$ per US$), from the cheapest to the most expensive selling rates:

   source       type  price_sell_usd
     Wise commercial        5.109600
  x-rates commercial        5.112301
DolarHoje    tourism        5.320000

>>> USD (tourism): DolarHoje at R$ 5.3200
